# Home-value inference

Load a trained checkpoint and predict from a Census-style record. This is an educational demo, not a production appraisal.

In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from generic_trainer.config import load_config
from models.housing_llm.factory import build_model
from models.housing_llm.tokenizer import StableHashTokenizer

In [ ]:
cfg = load_config(ROOT / 'configs/smoke.yaml')
model_cfg = {k: v for k, v in cfg['model'].items() if k != 'factory'}
model = build_model(model_cfg)
checkpoint_path = ROOT / 'artifacts/notebook-smoke/best.pt'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model'])
model.eval()

In [ ]:
record = {
    'state': 'NC', 'county': 'Wake', 'bedrooms': 3, 'bathrooms': 2,
    'living_area_sqft': 1850, 'median_income': 92000,
    'population': 1150000, 'year_built': 2004,
}
fields = cfg['data']['feature_fields']
prompt = 'Estimate median home value. ' + '; '.join(f'{k}={record[k]}' for k in fields)
tok_cfg = cfg['data']['tokenizer']
tokenizer = StableHashTokenizer(tok_cfg['vocab_size'], tok_cfg['max_length'])
ids, mask = tokenizer.encode(prompt)
with torch.inference_mode():
    output = model(torch.tensor([ids]), torch.tensor([mask]))
print(f"Predicted value: ${output['predictions'].item():,.0f}")